# 00 • Vérifier son environnement

`[MÉTA | Formation 4-024 | Niveau Application | TP 00 | Mode CPU local]`

**Objectif :** Diagnostiquer l’environnement et prouver qu’un calcul avec gradient fonctionne.

**Temps indicatif :** 25 min. Ces temps sont répartis dans le conducteur, pas additionnés hors des 18 heures.

**Prérequis :** Python élémentaire.

**Preuves de réussite :** Versions, calcul tensoriel, détection explicite des bibliothèques absentes.

**Sources :** R01, R02, R23 ; documentations officielles Keras, PyTorch, TensorFlow.

Les jeux métier sont synthétiques. Aucun fichier personnel ou fiscal réel ne doit être chargé. Les résultats obtenus ici ne constituent pas une validation métier.

**Mode d’emploi :** exécuter les cellules dans l’ordre. Les cellules d’exercice du cahier apprenant sont à compléter ; le corrigé contient le code et des résultats de référence sur CPU.

In [ ]:
from pathlib import Path
import sys, os, json
# Chercher le kit depuis le répertoire du notebook ou celui de lancement.
HERE = Path.cwd().resolve()
TP_ROOT = next((p for p in [HERE, *HERE.parents] if (p / "modules" / "atelier.py").exists()), None)
if TP_ROOT is None:
    raise FileNotFoundError("Ouvrir ce notebook depuis le dossier 03_Travaux_pratiques du kit décompressé.")
sys.path.insert(0, str(TP_ROOT / "modules"))
os.environ.setdefault("KERAS_BACKEND", "torch")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
from torch import nn
from atelier import *
seed_all(42)
print("Moteur disponible :", torch.__version__, "| Données :", DATA)


## 1. Inventaire reproductible
Un GPU accélère certains calculs. Il ne doit pas bloquer le socle de cette formation. Le kit a un chemin CPU. Ne pas mettre à jour les dépendances en milieu de session sans décision du formateur.

In [ ]:
import importlib.util, platform, importlib.metadata
packages = ["numpy", "pandas", "scikit-learn", "torch", "keras", "tensorflow", "matplotlib"]
versions = {name: (importlib.metadata.version(name) if importlib.util.find_spec({"scikit-learn":"sklearn"}.get(name,name)) else "absent") for name in packages}
rapport = {"python": platform.python_version(), "plateforme": platform.platform(), "versions":versions, "cuda":torch.cuda.is_available(), "backend_keras":os.environ["KERAS_BACKEND"]}
print(json.dumps(rapport,ensure_ascii=False,indent=2))
save_result("00_environnement",rapport)

## 2. Prouver le calcul du gradient
Pour L(w) = (w − 3)² et w = −4, la dérivée vaut 2 × (−4 − 3), soit −14. Calculer la valeur avec PyTorch. La création d’un gradient ne met pas encore le paramètre à jour.

In [ ]:
# EXERCICE À COMPLÉTER
# Créer un tenseur requires_grad=True, calculer la perte puis appeler backward().
# La correction est fournie séparément au formateur.
raise NotImplementedError("Compléter cette cellule puis relancer avant de poursuivre.")

## 3. Tester Keras et sa relation au moteur
Le moteur est choisi avant le premier import Keras. Dans ce kit hors ligne, le moteur par défaut est PyTorch. Dans un environnement préparé avec TensorFlow, redémarrer le noyau et définir KERAS_BACKEND=tensorflow avant tous les imports. Ne pas tenter de changer de moteur au milieu du notebook.

In [ ]:
import keras
keras.utils.set_random_seed(42)
model=keras.Sequential([keras.layers.Input((2,)),keras.layers.Dense(1)])
out=model(np.ones((3,2),dtype=np.float32))
assert tuple(out.shape)==(3,1)
print("Keras",keras.__version__,"moteur",keras.backend.backend(),"forme",tuple(out.shape))

## 4. Branche native TensorFlow, si installé
Ce test est distinct de celui de Keras. Son absence doit être signalée, jamais masquée comme une réussite. L’installation est une étape de préparation, pas une manipulation réseau automatique du notebook.

In [ ]:
if importlib.util.find_spec("tensorflow"):
    import tensorflow as tf
    variable=tf.Variable(-4.0)
    with tf.GradientTape() as tape:
        perte=(variable-3.0)**2
    grad=tape.gradient(perte,variable)
    assert float(grad.numpy())==-14.0
    print("TensorFlow testé :",tf.__version__,"gradient",float(grad.numpy()))
else:
    print("BRANCHE NON EXÉCUTÉE : TensorFlow absent. Le chemin PyTorch/Keras reste utilisable.")

## 5. Journal de préparation
Noter : bibliothèques disponibles ; bibliothèques absentes ; chemin des données ; mode CPU/GPU ; personne à prévenir en cas de blocage. **Critère d’acceptation :** aucune ambiguïté sur ce qui a été réellement testé.

**Extension :** lancer ce notebook depuis un noyau neuf et comparer l’inventaire. Une graine ne garantit pas une identité absolue entre versions et matériels.